In [1]:
import pandas as pd
import requests
import os
from IPython.display import display

# Datas de interesse
data_ini = '2005-01-01'
data_fim = '2026-07-10'

print("Extraindo múltiplas moedas do Banco Central Europeu...")

url = f"https://api.frankfurter.app/{data_ini}..{data_fim}?from=USD&to=CNY,EUR,TRY,BRL,INR"

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    
    dados = response.json()
    
    df_multi = pd.DataFrame.from_dict(dados['rates'], orient='index')
    df_multi.reset_index(inplace=True)
    
    df_multi.rename(columns={
        'index': 'Data', 
        'CNY': 'USD_CNY', 
        'EUR': 'USD_EUR', 
        'TRY': 'USD_TRY',
        'BRL': 'USD_BRL',
        'INR': 'USD_INR'
    }, inplace=True)
    
    # 1. Converte a coluna para o tipo datetime nativo do Pandas
    df_multi['Data'] = pd.to_datetime(df_multi['Data'])

    # 2. Ordena cronologicamente ENQUANTO ainda é formato de data
    df_multi.sort_values('Data', inplace=True)

    # 3. Só agora, com tudo na ordem certa, formata para texto com barras (DD/MM/YYYY)
    df_multi['Data'] = df_multi['Data'].dt.strftime('%d/%m/%Y')
    
    print("\nPré-visualização dos últimos registros:")
    display(df_multi.tail())
    
    # Salvando na pasta \dataset
    pasta_destino = 'dataset'
    nome_arquivo = 'ptax.xlsx'
    
    os.makedirs(pasta_destino, exist_ok=True)
    caminho_completo = os.path.join(pasta_destino, nome_arquivo)
    
    df_multi.to_excel(caminho_completo, index=False)
    print(f"\nArquivo salvo com sucesso em: {caminho_completo}")

except requests.exceptions.Timeout:
    print("Erro: A requisição demorou demais (Timeout).")
except Exception as e:
    print(f"Erro na extração: {e}")

Extraindo múltiplas moedas do Banco Central Europeu...

Pré-visualização dos últimos registros:


,Data,USD_BRL,USD_CNY,USD_EUR,USD_INR,USD_TRY
5505,06/07/2026,5.1800,6.7957,0.87604,95.40,46.800
5506,07/07/2026,5.1413,6.7935,0.87466,94.97,46.840
5507,08/07/2026,5.1563,6.8002,0.87689,95.56,46.857
5508,09/07/2026,5.1453,6.7960,0.87451,95.39,46.822
5509,10/07/2026,5.1185,6.7745,0.87489,95.34,46.985



Arquivo salvo com sucesso em: dataset\ptax.xlsx
